## **Here's a summary of the 75 cases generated:**

### **Distribution:**

HIGH RISK (27 cases): Cases 1, 4, 5, 6, 8, 9, 13, 15, 17, 18, 19, 20, 25, 28, 30, 32, 34, 36, 41, 42, 47, 48, 53, 54, 55, 58, 60, 65, 68, 70, 72, 75

MODERATE RISK (23 cases): Cases 2, 7, 11, 12, 16, 21, 23, 27, 29, 33, 35, 38, 39, 40, 43, 44, 45, 50, 52, 61, 66, 69, 73

LOW RISK (25 cases): Cases 3, 10, 14, 22, 24, 26, 31, 37, 46, 49, 51, 56, 57, 59, 62, 63, 64, 67, 71, 74

## **Key conditions covered:**

Eclampsia & severe preeclampsia (multiple cases, including postpartum)

PPH — primary (atony, home delivery) and secondary

Postpartum sepsis & endometritis

Gestational diabetes (mild to uncontrolled)

Severe & moderate anaemia

Ectopic pregnancy, placenta previa, abruption

Cord prolapse, PROM, chorioamnionitis

Neonatal sepsis, omphalitis, hypothermia, pathological jaundice

DVT, cardiac decompensation, suspected TB

Borderline cases (BP 142–150, borderline sugars, mild edema) to test model reasoning

## **Note**

All justifications cite clinical **thresholds aligned with MOHFW/WHO maternal referral guidelines** and community health danger sign protocols.

## **1 — Load Your Fine-Tuned Model (4-bit)**

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import torch

# 🔐 Login (required for gated MedGemma)
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

MODEL_ID = "google/medgemma-1.5-4b-it"
ADAPTER_PATH = "docvm/sakhi-medgemma-1.5-4b-maternal"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model.eval()

## **2 — Create Strict Evaluation Prompt**

In [ ]:
EVAL_SYSTEM_PROMPT = """
You are a maternal triage AI.

Classify the case into exactly one of:

Triage: HIGH
Triage: MODERATE
Triage: LOW

Escalate to HIGH if any of the following are present:
- BP ≥160 systolic or ≥110 diastolic
- Seizures, convulsions
- Heavy bleeding
- Signs of sepsis (fever + rigors + abdominal tenderness postpartum)
- Visual disturbance + hypertension
Otherwise classify appropriately.

Output strictly in this format:

Triageige: <HIGH/MODERATE/LOW>
Reason: <one short sentence>
"""

## **3 — Case → Prompt Builder**

In [ ]:
def build_prompt(case):
    return f"""
Age: {case['age']}
Gestational stage: {case['gestational_stage']}

Vitals:
- BP: {case['vitals']['bp']}
- Temperature: {case['vitals']['temperature']}
- Blood Sugar: {case['vitals']['blood_sugar']}

Symptoms: {case['symptoms']}
History: {case['history']}

Classify triage level.
"""

In [ ]:
import json

with open("/kaggle/input/datasets/vedantmisra108/maternal-triage-cases/maternal_triage_cases.json", "r") as f:
    cases = json.load(f)

print("Loaded cases:", len(cases))
print("First case preview:")
print(cases[0])

for case in cases:
    if case["guideline_triage"].upper() == "HIGH RISK":
        case["guideline_triage"] = "HIGH"
    elif case["guideline_triage"].upper() == "MODERATE RISK":
        case["guideline_triage"] = "MODERATE"
    elif case["guideline_triage"].upper() == "LOW RISK":
        case["guideline_triage"] = "LOW"

## **4 — Run Model on Each Case**

In [ ]:
import re
from tqdm import tqdm
import torch
import json

import re

def extract_structured_output(text):
    # Only look at content after the last "model"
    if "model" in text:
        text = text.split("model")[-1]

    triage_match = re.search(r"Triage:\s*(HIGH|MODERATE|LOW)", text, re.IGNORECASE)
    reason_match = re.search(r"Reason:\s*(.*)", text, re.IGNORECASE)

    triage = triage_match.group(1).upper() if triage_match else "UNKNOWN"
    reason = reason_match.group(1).strip() if reason_match else ""

    return triage, reason

results = []

for case in tqdm(cases):

    user_prompt = build_prompt(case)

    full_prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": EVAL_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.2
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    model_triage, model_reason = extract_structured_output(decoded)

    results.append({
        "case_id": case["case_id"],
        "prompt_used": user_prompt,
        "guideline_triage": case["guideline_triage"].upper(),
        "guideline_justification": case.get("justification", ""),
        "model_raw_output": decoded,
        "model_triage": model_triage,
        "model_reason": model_reason
    })

# Save full evaluation log
with open("sakhi_evaluation_log.json", "w") as f:
    json.dump(results, f, indent=2)

## **5 — Compute Metrics**

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

df = pd.DataFrame(results)

labels = ["HIGH", "MODERATE", "LOW"]

cm = confusion_matrix(
    df["guideline_triage"],
    df["model_triage"],
    labels=labels
)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("Confusion Matrix:")
print(cm_df)

### **Compute Key Metrics**

In [ ]:
overall_agreement = (df["guideline_triage"] == df["model_triage"]).mean()
print("Overall Agreement:", round(overall_agreement * 100, 2), "%")

In [ ]:
high_cases = df[df["guideline_triage"] == "HIGH"]
false_negatives = high_cases[high_cases["model_triage"] != "HIGH"]

fnr = len(false_negatives) / len(high_cases)
print("False Negative Rate (HIGH):", round(fnr * 100, 2), "%")

In [ ]:
predicted_high = df[df["model_triage"] == "HIGH"]
false_positives = predicted_high[predicted_high["guideline_triage"] != "HIGH"]

fpr = len(false_positives) / len(df[df["guideline_triage"] != "HIGH"])
print("False Positive Rate (HIGH):", round(fpr * 100, 2), "%")

In [ ]:
TP = cm_df.loc["HIGH", "HIGH"]
FN = cm_df.loc["HIGH"].sum() - TP
FP = cm_df["HIGH"].sum() - TP
TN = cm_df.values.sum() - TP - FN - FP

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

print("Sensitivity (HIGH):", round(sensitivity * 100, 2), "%")
print("Specificity:", round(specificity * 100, 2), "%")

### **Error Pattern Analysis**

In [ ]:
errors = df[df["guideline_triage"] != df["model_triage"]]
print(errors.head())

In [ ]:
print(df["model_triage"].value_counts())
print(df["guideline_triage"].value_counts())

In [ ]:
for r in results[:10]:
    print("GOLD:", r["guideline_triage"])
    print("MODEL RAW:", r["model_raw_output"])
    print("PARSED:", r["model_triage"])
    print("-----")